# Dynamic Arrays

```{contents}
:local:
:depth: 2
```

A dynamic array gives the programmer the feel of a growable sequence while storing values in ordinary contiguous array storage behind the scenes.


```{index} contiguous storage
```
```{index} dynamic array
```



```{index} Contiguous Storage
```

## Contiguous Storage



An array stores elements in numbered positions. If the array has five slots, valid indexes are `0` through `4`.

```text
index:  0   1   2   3   4
value: [10][20][30][  ][  ]
```

The array has a fixed **capacity**: the number of slots available. A dynamic array also tracks **count**: the number of slots currently being used.

Two numbers describe the structure at every moment. *Capacity* is `storage.Length`, fixed when the array was created. *Count* is how many of those slots hold real data, and it is the program's job to track it, because the array itself cannot tell a stored `0` from an unused slot. The append pattern is always the same two steps:

```csharp
storage[count] = value;   // write into the first unused slot
count++;                  // that slot is now used
```

The example appends three values and reads one back by index:


In [ ]:
int[] storage = new int[5];
int count = 0;

storage[count] = 10;
count++;
storage[count] = 20;
count++;
storage[count] = 30;
count++;

Console.WriteLine($"count={count}, capacity={storage.Length}");
Console.WriteLine(storage[1]);


count=3, capacity=5
20


The output is `count=3, capacity=5` and then `20`. Slots 3 and 4 still hold the default `0`, but they are not part of the sequence: a loop over the data must stop at `count`, not at `storage.Length`. Using `Length` as the number of items is the first mistake to avoid; it would report five values, two of them phantom zeros. The second is appending when `count == storage.Length`, which throws `IndexOutOfRangeException`; the next section shows what to do instead.


```{index} capacity
```
```{index} resizing; dynamic array
```



```{index} Resizing by Copying
```

## Resizing by Copying



When the array fills up, the dynamic array cannot stretch the existing array. It allocates a larger array, copies the existing values, then continues adding into the new storage.

A common strategy is to double the capacity each time resizing is needed. This makes individual resize steps expensive, but keeps most `Add` operations cheap.

The resize has three steps, and the order matters: allocate the larger array, copy every used slot, then make the variable refer to the new array. The old array is unreachable afterward and is reclaimed by the runtime.

```csharp
int[] larger = new int[old.Length * 2];   // 1. allocate
for (int i = 0; i < old.Length; i++)      // 2. copy the used slots
    larger[i] = old[i];
storage = larger;                         // 3. the variable now refers to the new array
```

`Grow` below performs the first two steps and returns the result; the caller performs the third.


In [ ]:
static int[] Grow(int[] oldStorage)
{
    int[] larger = new int[oldStorage.Length * 2];
    for (int i = 0; i < oldStorage.Length; i++)
    {
        larger[i] = oldStorage[i];
    }
    return larger;
}

int[] storage = new int[5];
storage[0] = 10;
storage[1] = 20;
storage[2] = 30;
Console.WriteLine(string.Join(", ", storage));
Console.WriteLine($"Length of storage: {storage.Length}");

Console.WriteLine();

int[] newStorage = Grow(storage);
newStorage[2] = 30;

Console.WriteLine(string.Join(", ", newStorage));
Console.WriteLine($"Length of new storage: {newStorage.Length}");

10, 20, 30, 0, 0
Length of storage: 5

10, 20, 30, 0, 0, 0, 0, 0, 0, 0
Length of new storage: 10


The first two lines print `10, 20, 30, 0, 0` with `Length of storage: 5`; after the blank line the new array prints `10, 20, 30, 0, 0, 0, 0, 0, 0, 0` with `Length of new storage: 10`. The three stored values arrived in the larger array through the copy loop, so the assignment `newStorage[2] = 30` is redundant: it rewrites a value that was already copied. The mistake that matters is forgetting step three. If the dynamic array called `Grow(storage)` without assigning the result back to its field, it would keep appending into the old, full array and fail on the next add.


```{index} Guided Practice: Append with Growth
```

### Guided Practice: Append with Growth


Append seven values into storage that starts with capacity 2. The starter throws `IndexOutOfRangeException` at the third append because it never grows; add the growth check so the output is `count=7, capacity=8`.


In [ ]:
using System;

int[] storage = new int[2];
int count = 0;

for (int value = 10; value <= 70; value += 10)
{
    // Your code starts here.
    // If storage is full, replace it with a copy of double the length.
    // Your code ends here.
    storage[count] = value;
    count++;
}

Console.WriteLine($"count={count}, capacity={storage.Length}");


In [ ]:
using System;

// Solution
int[] storage = new int[2];
int count = 0;

for (int value = 10; value <= 70; value += 10)
{
    if (count == storage.Length)
    {
        int[] larger = new int[storage.Length * 2];
        Array.Copy(storage, larger, count);
        storage = larger;        // the variable must refer to the new array
    }
    storage[count] = value;
    count++;
}

Console.WriteLine($"count={count}, capacity={storage.Length}");


count=7, capacity=8


Capacity went 2, 4, 8, with a copy at the third and fifth appends. Leave out the line `storage = larger;` and the program still throws at the third append: the copy was made, but the loop kept writing into the old array.


```{index} indexed access
```



```{index} Worked Example: A Small Dynamic Array
```

## Worked Example: A Small Dynamic Array



This class stores integers in an internal array. `Add` appends to the next free slot. The indexer reads an existing value by index.

Read `Add` in two halves. The `if` handles the full case by doubling, exactly as `Grow` did, but assigns the larger array back to the field so the object keeps it. The last two lines are the two-step append pattern. The indexer checks `index` against `Count`, not against `items.Length`, so unused capacity is never readable. Starting capacity is 2, so the five appends below resize twice; the printed line after each append shows when.


In [ ]:
var values = new DynamicIntArray();
for (int value = 10; value <= 50; value += 10)
{
    values.Add(value);
    Console.WriteLine($"added={value}, count={values.Count}, capacity={values.Capacity}");
}

Console.WriteLine($"values[3]={values[3]}");

public class DynamicIntArray
{
    private int[] items = new int[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;

    public void Add(int value)
    {
        if (Count == items.Length)
        {
            int[] larger = new int[items.Length * 2];
            Array.Copy(items, larger, items.Length);
            items = larger;
        }

        items[Count] = value;
        Count++;
    }

    public int this[int index]
    {
        get
        {
            if (index < 0 || index >= Count)
                throw new IndexOutOfRangeException();
            return items[index];
        }
    }
}


added=10, count=1, capacity=2
added=20, count=2, capacity=2
added=30, count=3, capacity=4
added=40, count=4, capacity=4
added=50, count=5, capacity=8
values[3]=40


The output is:

```text
added=10, count=1, capacity=2
added=20, count=2, capacity=2
added=30, count=3, capacity=4
added=40, count=4, capacity=4
added=50, count=5, capacity=8
values[3]=40
```

Capacity doubled on the third and fifth appends, when `Count` had caught up with it. After the last line, `values[4]` is `50` and `values[5]` throws, even though the backing array has eight slots: the indexer enforces the sequence contract, not the storage size. Variation: change the starting capacity to 1 and predict the capacity column before running.


```{index} amortized cost
```



```{index} Cost of Append and Indexing
```

## Cost of Append and Indexing



Dynamic arrays have two different stories:

- Reading `items[index]` is O(1) because the index points directly to the slot.
- Most appends are O(1), but an append that triggers resizing is O(n) because it copies existing values.

Over many appends, doubling capacity gives **amortized O(1)** append cost. Amortized means the occasional expensive resize is spread across many cheap operations.

Count the copies for sixteen appends starting from capacity 2:

| Resize at append | New capacity | Values copied |
| ---: | ---: | ---: |
| 3 | 4 | 2 |
| 5 | 8 | 4 |
| 9 | 16 | 8 |
| Total | | 14 |

Sixteen appends caused fourteen copies, fewer than one per append, and the ratio stays below one however long the sequence grows. That is the amortized argument in numbers; the last section of this page states it in general.

The library's `List<T>` follows the same policy, so the two stories apply to every list you have used since Chapter 8: `list[i]` is a direct read, `list.Add(x)` is usually one write, and every so often an `Add` pays for a copy of the whole list. `List<T>.Capacity` exposes the current array length, and a program that knows how many items are coming can set it in the constructor, `new List<T>(10000)`, to avoid the resizes entirely. That is the one tuning step worth remembering: it changes constants, not the growth class.

```{index} Code Tracing: Capacity After Each Append
```

### Code Tracing: Capacity After Each Append

A dynamic array starts with capacity 3 and doubles when full. Without running anything, fill in the table for appends 1 through 10: the count after the append, the capacity after it, and how many existing values were copied during it. Then predict the two printed lines of the program below, which reports the capacity after the tenth append and the total number of copies.

| Append | Count after | Capacity after | Values copied |
| ---: | ---: | ---: | ---: |
| 1 | 1 | 3 | 0 |
| 2 | | | |
| ... | | | |
| 10 | | | |

```csharp
int capacity = 3, count = 0, copies = 0;
for (int append = 1; append <= 10; append++)
{
    if (count == capacity) { copies += count; capacity *= 2; }
    count++;
}
Console.WriteLine($"capacity={capacity}");
Console.WriteLine($"copies={copies}");
```

In [ ]:
// Solution
using System;

int capacity = 3, count = 0, copies = 0;
for (int append = 1; append <= 10; append++)
{
    if (count == capacity) { copies += count; capacity *= 2; }   // appends 4 and 7 resize
    count++;
    Console.WriteLine($"append {append,2}: count={count,2} capacity={capacity,2} copies so far={copies}");
}
Console.WriteLine($"capacity={capacity}");
Console.WriteLine($"copies={copies}");

append  1: count= 1 capacity= 3 copies so far=0
append  2: count= 2 capacity= 3 copies so far=0
append  3: count= 3 capacity= 3 copies so far=0
append  4: count= 4 capacity= 6 copies so far=3
append  5: count= 5 capacity= 6 copies so far=3
append  6: count= 6 capacity= 6 copies so far=3
append  7: count= 7 capacity=12 copies so far=9
append  8: count= 8 capacity=12 copies so far=9
append  9: count= 9 capacity=12 copies so far=9
append 10: count=10 capacity=12 copies so far=9
capacity=12
copies=9


The resizes happen on the fourth append (3 copies, capacity 6) and the seventh (6 copies, capacity 12), so the final capacity is 12 and the total is 9 copies for 10 appends. Appends 8, 9, and 10 fit without copying. If your table resized on append 3, you tested `count == capacity` after incrementing rather than before: the array is full when three values occupy three slots, and the *next* append is the one that must grow it.

```{index} sequence; invariant; Count; Capacity
```



```{index} A Sequence Contract and Its Invariants
```

## A Sequence Contract and Its Invariants



A service desk keeps work orders in processing order. The values might be strings such as `WO-10` and `WO-30`, but the representation should work for other types too. Chapter 13 separated an ADT's behavior from its implementation. Here the behavior is an ordered sequence: append a work order, read its position, insert an urgent order, and remove a completed order.

For an array-backed implementation, three facts must remain true after every public operation:

1. `0 <= Count <= Capacity`.
2. The live sequence occupies indexes `0` through `Count - 1` without gaps.
3. Values outside that range are storage, not members of the sequence.

A zero in an integer array is not a reliable empty marker: zero might be a real value. `Count` determines membership. With `Count = 3` and `Capacity = 5`, reading index `4` must fail even though the underlying array physically has that slot. An insertion at index `3` is valid because it appends; a read at index `3` is invalid.

**Check your understanding (basic):** An array has capacity eight and count three. List the legal read indexes and the legal insertion indexes. Explain why these sets differ.

```{index} Index-Based and Position-Based Designs
```

### Index-Based and Position-Based Designs

The contract above locates elements by integer index, which is the design the C# library uses for `List<T>`. A classic alternative keeps a *current position* inside the list object itself: operations such as `MoveToStart`, `Next`, `Prev`, `Insert`, and `Remove` all act at that position, and the client never supplies an index. The two designs promise the same sequence but distribute work differently.

| Concern | Index-based (this chapter, `List<T>`) | Current-position (cursor) design |
| --- | --- | --- |
| How a client names a place | An integer the client computes and passes | A position the list remembers between calls |
| Insert in the middle of an array | Shift once, starting from the supplied index | Shift once, starting from the stored position |
| Insert in the middle of a linked chain | Traverse to the index, then relink | Relink at the cursor; the traversal was paid earlier by `Next` calls |
| Several places at once | Pass several indexes | Needs several cursor objects, which is the iterator design of Chapter 13 |
| Risk | Index drifts after insertions and removals | A cursor left at a removed element is undefined |

Neither design changes the costs table: the array still shifts and the chain still traverses. What the cursor design makes visible is that *locating* a place and *updating* at that place are separate steps, a distinction the lab in 14.3 depends on. The library separates the two in a different way, by handing out enumerators for walking and indexes for updating; the `LinkedList<T>` class even exposes node handles so that a client holding one can insert next to it without a search.

```{index} dynamic array; resize trace; alias
```



```{index} Trace Growth Before Writing Code
```

## Trace Growth Before Writing Code



Start with capacity two. Appending `WO-10` and `WO-30` uses both slots. Before a third append, allocate four slots, copy the two live references, and replace the private storage reference. The logical sequence has not changed yet. Only then write the new value and increase `Count`.

```text
Before growth: items -> [WO-10][WO-30]                 Count = 2
New storage:   larger -> [WO-10][WO-30][unused][unused]
After growth:  items -> [WO-10][WO-30][unused][unused]   Count = 2
After append:  items -> [WO-10][WO-30][WO-40][unused]    Count = 3
```

For reference-type elements, copying the array copies references, not the referenced objects. The work order objects would retain their identity. Our string labels let you concentrate on the sequence rather than object mutation.

| Append number | Count after append | Capacity | Existing entries copied |
| ---: | ---: | ---: | ---: |
| 1 | 1 | 2 | 0 |
| 2 | 2 | 2 | 0 |
| 3 | 3 | 4 | 2 |
| 4 | 4 | 4 | 0 |
| 5 | 5 | 8 | 4 |
| 6 | 6 | 8 | 0 |
| 7 | 7 | 8 | 0 |
| 8 | 8 | 8 | 0 |
| 9 | 9 | 16 | 8 |

**Predict (basic):** Before reading on, give the capacity and copy count for append 10. Does it resize again?

```{index} dynamic array; insertion; removal; shifting
```



```{index} Middle Updates: Preserve the Unread Values
```

## Middle Updates: Preserve the Unread Values



Suppose the live values are `[10, 20, 40, 50]`, and you must insert `30` at index two. First ensure space for a fifth value. Then shift from the right end toward the insertion point:

| Step | Live values and destination slot | Reason |
| --- | --- | --- |
| Start | `[10, 20, 40, 50, unused]` | Index two must become available. |
| Copy index 3 to 4 | `[10, 20, 40, 50, 50]` | Save the last value first. |
| Copy index 2 to 3 | `[10, 20, 40, 40, 50]` | The old index three is already safe. |
| Write 30 at index 2 | `[10, 20, 30, 40, 50]` | Increase count after storing the new value. |

Shifting left to right would overwrite values you have not copied yet. The temporary duplicates in the trace are expected; the sequence invariant applies at the operation boundary.

Removal moves in the opposite direction. To remove index one from `[10, 20, 30, 40]`, save `20`, copy `30` left, then copy `40` left. Reduce count to three and clear the formerly live final slot. Clearing prevents a removed reference from being retained unnecessarily; it does not reduce capacity.

Insertion at index `i` moves `n - i` existing entries. Removal moves `n - i - 1`. Appending and removing the last item move no entries, although an append may still resize.

**Trace (applied):** Insert at index zero and then remove the last item. Record each assignment. Explain why reversing the insertion loop corrupts the original sequence.

```{index} generics; dynamic array; bounds checking
```



```{index} Complete Generic Implementation
```

## Complete Generic Implementation



The following implementation gives the work-order sequence a reusable element type `T`. `EnsureRoom` handles physical storage; `Insert` handles logical order. Keeping these responsibilities separate means growth cannot accidentally change count.

`default!` clears the unused slot. The exclamation mark suppresses a nullable-analysis warning for that storage assignment; it does not check anything at runtime. Clients cannot read the cleared slot because the indexer checks `Count`.

Run this cell independently. Expect count three, capacity four, the labels in order, and then removal of `WO-10`. This is an instructional implementation: it omits shrinking, iterators, concurrency, and protection against enormous capacity overflow.

In [ ]:
using System;
var orders = new WorkOrderArray<string>();
orders.Add("WO-10");
orders.Add("WO-30");
orders.Insert(1, "WO-20");
Console.WriteLine($"count={orders.Count}, capacity={orders.Capacity}");
for (int i = 0; i < orders.Count; i++) Console.WriteLine(orders[i]);
Console.WriteLine($"removed={orders.RemoveAt(0)}");
for (int i = 0; i < orders.Count; i++) Console.WriteLine(orders[i]);
public class WorkOrderArray<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public T this[int index]
    {
        get { CheckElement(index); return items[index]; }
    }
    private void CheckElement(int index)
    {
        if (index < 0 || index >= Count)
            throw new ArgumentOutOfRangeException(nameof(index));
    }
    private void EnsureRoom()
    {
        if (Count < Capacity) return;
        var larger = new T[Capacity * 2];
        Array.Copy(items, larger, Count);
        items = larger;
    }
    public void Add(T value) => Insert(Count, value);
    public void Insert(int index, T value)
    {
        if (index < 0 || index > Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        EnsureRoom();
        for (int i = Count; i > index; i--)
            items[i] = items[i - 1];
        items[index] = value;
        Count++;
    }
    public T RemoveAt(int index)
    {
        CheckElement(index);
        T removed = items[index];
        for (int i = index; i < Count - 1; i++)
            items[i] = items[i + 1];
        Count--;
        items[Count] = default!;
        return removed;
    }
}

count=3, capacity=4
WO-10
WO-20
WO-30
removed=WO-10
WO-20
WO-30


Read the implementation in operation order. `Insert` validates before mutation, grows before shifting, moves right to left, writes the new value, and increments count once. `RemoveAt` validates before mutation, saves the result, compacts left to right, decreases count once, and clears the unused slot.

**Modify (applied):** Add `Contains(T value)` using `EqualityComparer<T>.Default`. Test repeated values and a missing value. Explain why supporting generics does not make searching constant-time.

```{index} amortized analysis; worst case; dynamic array
```



```{index} Why Doubling Gives Amortized Constant Cost
```

## Why Doubling Gives Amortized Constant Cost



For nine appends, the trace copies `2 + 4 + 8 = 14` existing entries and writes nine new entries. Counting these array writes gives 23 writes across nine operations. Some individual appends are costly, but the total grows in proportion to the number of appends.

For a longer sequence, copy sizes form `2 + 4 + 8 + ...`. Each term is twice the previous one, so the sum stays below twice the final number of inserted items. Adding one write per appended item still gives a total proportional to `n`. Dividing total work by `n` gives a constant amortized cost. This is a bound over an operation sequence, not an assumption about random inputs.

Growing by one slot each time would copy `2 + 3 + 4 + ... + (n - 1)` entries. That total grows quadratically. Avoiding spare capacity would therefore make repeated appends progressively expensive.

The tradeoff is spare storage. Doubling leaves unused slots after growth, and a resize temporarily keeps both old and new arrays alive. Amortized O(1) also does not guarantee a short pause for every append. A system with a strict response deadline may reserve capacity ahead of time.

**Explain (challenge):** Compare doubling with adding one slot using ten appends starting at capacity two. Count copies, not elapsed milliseconds. State both the worst-case cost of one append and the amortized cost of the whole sequence.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. `values.Count` is 6 and `values.Capacity` is 8. How many slots of the backing array are legal to read through the indexer, and what happens on `values[6]`?

```{dropdown} Answer
Six slots, indexes 0 through 5. Reading index 6 throws even though the array physically has that slot: the indexer checks against `Count`, because slots 6 and 7 hold storage, not members of the sequence.
```

2. A program appends one million items to a dynamic array that starts at capacity 1 and doubles. About how many resizes occur, and about how many values are copied in total?

```{dropdown} Answer
About 20 resizes, because 2 to the 20th is just over a million. The copies form the series 1 + 2 + 4 + ... up to about half a million, which sums to just under one million: fewer than one copy per appended item, which is the amortized argument.
```

3. Why is growing by a fixed 10 slots at a time worse than doubling for long sequences, even though it wastes less space?

```{dropdown} Answer
Each resize still copies every stored value, and with a fixed increment the number of resizes grows in proportion to n rather than to log n. The total copying becomes quadratic in n. Doubling keeps the resizes rare enough that total copying stays linear.
```

4. Inserting at index 0 of a 5,000-item dynamic array and appending to it are both single operations. Why do their costs differ by a factor of thousands?

```{dropdown} Answer
Appending writes one slot. Inserting at the front must shift all 5,000 existing values one slot to the right to open the gap, so it does 5,000 writes before the new value is stored. The costs table records this as O(n) for middle and front insertion against O(1) amortized for append.
```

```{index} practice
```



```{index} Practice
```

## Practice



1. Trace the capacities after appending six values to a dynamic array that starts with capacity `2` and doubles when full.
2. Explain why indexed access is fast in an array-backed representation.
3. Add a `Contains(int value)` method to `DynamicIntArray`. What is its worst-case cost?


```{index} summary
```



```{index} Summary
```

## Summary



A dynamic array combines contiguous storage with a resizing policy. It is strong when a program needs fast indexing and frequent appends, but middle insertion and deletion require shifting elements.


```{rubric} Footnotes
```


[^1401_dynamic_arrays-1]: C# `List<T>` is implemented as a dynamic array conceptually similar to this simplified example, with many additional features and optimizations.
